# Laika MoE para a HP 50g

Mistura de especialistas (~9,7M de parâmetros no total, ~2,5M ativos por token) para rodar em C na HP 50g, com o
mesmo tokenizador do modelo atual. Tudo fica em `Meu Drive/LLM_Laika/hp50g/`. Detalhes em `hp50g/LEIA-ME.md`.

Ordem: 1 → 2 (sem GPU) → 3 (varredura, A100) → 4 (denso de comparação, A100) → 6 (treino final, A100) → 7 (exportar,
sem GPU). Se uma célula de treino parar no limite de ~6 h, rode de novo com os mesmos valores: continua de onde parou.

In [ ]:
# 1) PREPARAR: GPU, Google Drive, código e atalhos
RAMO = "main"  #@param {type:"string"}
import os, subprocess, codecs, json, glob
def rodar(comando):
    print("$", comando, flush=True)
    p = subprocess.Popen(comando, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env=dict(os.environ, PYTHONUNBUFFERED="1"))
    t = codecs.getincrementaldecoder("utf-8")("replace")
    while pedaco := p.stdout.read1(65536):
        print(t.decode(pedaco), end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"o comando falhou (código {p.returncode}): {comando}")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (só processador)")
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/LLM_Laika"
HP = f"{DRIVE}/hp50g"
DADOS = f"{HP}/dados_moe"  # os dados do MoE (os de hp50g/dados ficam como estão)
os.makedirs(HP, exist_ok=True)
if not os.path.exists("/content/LLM_Laika"):
    rodar("git clone -q https://github.com/BrunoDevPro/LLM_Laika.git /content/LLM_Laika")
os.chdir("/content/LLM_Laika")
rodar(f"git fetch -q origin {RAMO} && git checkout -q -f -B {RAMO} FETCH_HEAD")
rodar("pip install -q tokenizers")
# Os dados que a Laika já baixou (só se for preciso montar o texto simples de novo) e o SFT.
for pasta in ["wiki", "gigaverbo", "carolina", "sft"]:
    local, remoto = f"data/{pasta}", f"{DRIVE}/{pasta}"
    if os.path.isdir(remoto) and not os.path.islink(local):
        rodar(f"rm -rf {local}")
        os.symlink(remoto, local)
LOCAL = "/content/hp50g_dados"
def dados_locais():
    """Cópia local dos .bin/.mask/.json (ler do Drive durante o treino é lento)."""
    rodar(f"mkdir -p {LOCAL} && rsync -a --include='*.bin' --include='*.mask' --include='*.json' --exclude='*' {DADOS}/ {LOCAL}/")
    return LOCAL

In [ ]:
# 2) DADOS DO MoE (sem GPU): o mesmo tokenizador, perguntas como no teclado da 50g, respostas curtas
# TOKENIZADOR: o tokenizador.json do modelo que já roda na calculadora (o vocab.bin não muda). Vazio = o da exportação
# A_final, se existir; senão, o de hp50g/dados.
# NORMALIZAR: fração das perguntas sem acentos, como chegam do teclado (a resposta continua acentuada).
# RESPOSTA_CURTA: respostas encurtadas para até tantos caracteres (frases inteiras); cada token custa ~2 s na 50g.
# ARQUIVO_FATOS: pares "pergunta|resposta" para decorar (opcional), repetidos REPETIR_FATOS vezes.
# O texto simples é o de hp50g/dados (texto_treino.txt); se não existir, é montado de novo a partir dos dados da Laika.
TOKENIZADOR = ""  #@param {type:"string"}
NORMALIZAR = 0.7  #@param {type:"number"}
RESPOSTA_CURTA = 60  #@param {type:"integer"}
ARQUIVO_FATOS = "/content/drive/MyDrive/LLM_Laika/hp50g/meus_fatos.txt"  #@param {type:"string"}
REPETIR_FATOS = 8  #@param {type:"integer"}
TEXTO_MB = 1500  #@param {type:"integer"}
tok = TOKENIZADOR.strip() or next((t for t in (f"{HP}/export/A_final/tokenizador.json", f"{HP}/dados/tokenizador.json")
                                   if os.path.exists(t)), "")
if not tok:
    raise RuntimeError("Não achei o tokenizador do modelo atual: informe TOKENIZADOR (o tokenizador.json da exportação).")
print("Tokenizador:", tok)
extra = f" --texto-de {HP}/dados" if os.path.exists(f"{HP}/dados/texto_treino.txt") else f" --texto-mb {TEXTO_MB}"
if ARQUIVO_FATOS.strip() and os.path.exists(ARQUIVO_FATOS.strip()):
    extra += f" --decorar '{ARQUIVO_FATOS.strip()}' --decorar-rep {REPETIR_FATOS}"
rodar(f"python hp50g/preparar.py --saida {DADOS} --usar-tokenizador '{tok}' --normalizar {NORMALIZAR} "
      f"--resposta-curta {RESPOSTA_CURTA} --max-trocas 1{extra}")

In [ ]:
# 3) VARREDURA CURTA (A100): taxa de aprendizado do MoE, com 100–200 milhões de tokens cada
VARIANTES = "MoE"  #@param {type:"string"}
TAXAS = "1e-3,2e-3,3e-3"  #@param {type:"string"}
TOKENS = "150 milhões"  #@param ["100 milhões", "150 milhões", "200 milhões"]
n = {"100 milhões": 1e8, "150 milhões": 1.5e8, "200 milhões": 2e8}[TOKENS]
dl = dados_locais()
for v in VARIANTES.split(","):
    for lr in TAXAS.split(","):
        saida = f"{HP}/varredura/{v.strip()}_lr{lr.strip()}_{int(n / 1e6)}M"
        if os.path.exists(f"{saida}/resumo.json"):
            continue
        rodar(f"python hp50g/treino.py --variante {v.strip()} --lr {lr.strip()} --tokens {n:.0f} --dados {dl} "
              f"--saida {saida} --compile")
print("| execução | perda float | perda quantizada |\n|---|---|---|")
for r in sorted(glob.glob(f"{HP}/varredura/*/resumo.json")):
    j = json.load(open(r))
    print(f"| {r.split('/')[-2]} | {j['melhor_perda_float']:.4f} | {j['melhor_perda_quant']:.4f} |")

In [ ]:
# 4) DENSO DE COMPARAÇÃO (A100): ~2,5M de parâmetros, o mesmo custo por token do MoE, nos mesmos dados
# Se o MoE não ganhar dele na perda quantizada, o denso é a escolha (a célula 7 compara os dois).
TAXA = 3e-3  #@param {type:"number"}
TOKENS = 2.5e9  #@param {type:"number"}
NOME = "denso_2.5M"  #@param {type:"string"}
dl = dados_locais()
rodar(f"python hp50g/treino.py --variante denso_2.5M --lr {TAXA} --tokens {TOKENS:.0f} --dados {dl} "
      f"--saida {HP}/final/{NOME} --compile")

In [ ]:
# 5) PROFESSOR
print("Pulado (a destilação é opcional e não está no plano do MoE).")

In [ ]:
# 6) TREINO FINAL DO MoE (A100): ~2,5 bilhões de tokens; QAT e perda só nas respostas nos últimos 15%
# TAXA: a melhor da varredura (célula 3). Se parar no limite de tempo, rode de novo com os mesmos valores.
VARIANTE = "MoE"  #@param {type:"string"}
TAXA = 2e-3  #@param {type:"number"}
TOKENS = 2.5e9  #@param {type:"number"}
NOME = "MoE_final"  #@param {type:"string"}
DO_ZERO = False  #@param {type:"boolean"}
dl = dados_locais()
rodar(f"python hp50g/treino.py --variante {VARIANTE} --lr {TAXA} --tokens {TOKENS:.0f} --dados {dl} "
      f"--saida {HP}/final/{NOME} --compile" + (" --do-zero" if DO_ZERO else ""))

In [ ]:
# 7) EXPORTAR E RELATÓRIO (sem GPU): hplm.bin (v2 no MoE), quant.json, relatorio.json, vocab.bin, vetores_teste.json
# EXECUCOES: as pastas de hp50g/final a exportar, separadas por vírgula. COMPARAR: o denso de comparação (exportado
# antes, e a perda dele entra no relatório do MoE). JANELAS: janelas de validação no simulador (50 já basta).
EXECUCOES = "denso_2.5M,MoE_final"  #@param {type:"string"}
COMPARAR = "denso_2.5M"  #@param {type:"string"}
JANELAS = 50  #@param {type:"integer"}
VETORES = 50  #@param {type:"integer"}
nomes = [n.strip() for n in EXECUCOES.split(",") if n.strip()]
nomes.sort(key=lambda n: n != COMPARAR.strip())  # o de comparação primeiro
pacotes = []
for n in nomes:
    pasta = next((p for p in (f"{HP}/final/{n}", f"{HP}/varredura/{n}") if os.path.isdir(p)), None)
    if pasta is None:
        print("Não achei", n)
        continue
    comp = f" --comparar {HP}/export/{COMPARAR.strip()}" if COMPARAR.strip() and n != COMPARAR.strip() else ""
    rodar(f"python hp50g/simulador.py exportar --ckpt {pasta}/melhor.pt --dados {DADOS} --saida {HP}/export/{n} "
          f"--nome {n} --janelas {JANELAS} --vetores {VETORES}{comp}")
    pacotes.append(f"{HP}/export/{n}")
rodar(f"python hp50g/simulador.py relatorio --pacotes {' '.join(pacotes)} --saida {HP}/export/relatorio.md")